In [1]:
import torch
import torch.nn as nn
import os
import pandas as pd
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm import tqdm
import os
import random
import json
import shutil
from sklearn.model_selection import StratifiedKFold
import cv2

In [ ]:
class FUSegBaseline(Dataset):
    def __init__(self, root, split='train', transform=None):
        self.imgs = sorted((Path(root) / split / 'images').glob('*.png'))
        self.masks = [Path(root)/split/'labels'/f.name for f in self.imgs]
        self.transform = transform
        
    def __len__(self): return len(self.imgs)
        
    def __getitem__(self, i):
        img = torch.from_numpy(np.array(Image.open(self.imgs[i]).convert('RGB'))).permute(2,0,1).float()
        mask = torch.from_numpy(np.array(Image.open(self.masks[i]).convert('L')) > 127).long()
        if self.transform:
            aug = self.transform(image=img.numpy().transpose(1,2,0), mask=mask.numpy())
            img, mask = torch.from_numpy(aug['image'].transpose(2,0,1)).float(), torch.from_numpy(aug['mask']).long()
            
        return {'image': img, 'mask': mask}

In [3]:
class DiceFocalLoss(nn.Module):
    """
    Комбинация Dice Loss + Focal Loss (как в FUSeg Rank 1).

    Args:
        dice_weight: вес Dice-компоненты 
        focal_weight: вес Focal-компоненты
        focal_gamma: параметр фокусировки
        focal_alpha: баланс классов для Focal
    """
    def __init__(self, dice_weight=1.0, focal_weight=0.5,
                 focal_gamma=2.0, focal_alpha=0.25):
        super().__init__()
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight
        self.dice_loss = smp.losses.DiceLoss(mode='binary', smooth=1.0)
        self.focal_loss = smp.losses.FocalLoss(
            mode='binary',
            gamma=focal_gamma,
            alpha=focal_alpha
        )

    def forward(self, logits, targets):
        targets = targets.float()
        loss_dice = self.dice_loss(torch.sigmoid(logits), targets)
        loss_focal = self.focal_loss(logits, targets)
        return self.dice_weight * loss_dice + self.focal_weight * loss_focal

In [4]:
if torch.cuda.is_available():
    print(f"Имя вашей видеокарты: {torch.cuda.get_device_name(0)}")
    print(f"Количество видеокарт: {torch.cuda.device_count()}")

Имя вашей видеокарты: NVIDIA GeForce RTX 4070 Laptop GPU
Количество видеокарт: 1


In [16]:
# Аугментации
random.seed(42)

train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.85, 1.15), rotate=15, translate_percent=0.05, p=0.5),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
    A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05, p=0.4),
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
], additional_targets={'mask': 'mask'})

val_tf = A.Compose([
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
], additional_targets={'mask':'mask'})

# Лоадеры
DATASET_ROOT = Path("../data/raw/Foot Ulcer Segmentation Challenge")
train_ds = FUSegBaseline(DATASET_ROOT, 'train', train_tf)
val_ds   = FUSegBaseline(DATASET_ROOT, 'validation', val_tf)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

In [17]:
# Model, Loss, Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = smp.Unet(encoder_name='timm-efficientnet-b0', encoder_weights='imagenet', in_channels=3, classes=1, activation=None).to(device)
criterion = DiceFocalLoss(
    dice_weight=1.0,      # Dice — основная метрика
    focal_weight=0.5,     # Focal — вспомогательная стабилизация
    focal_gamma=2.0,      # Стандартное значение из оригинальной статьи
    focal_alpha=0.25      # Усиливает штраф за пропуск раны (класс 1)
)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=4, factor=0.5)
print(device)

cuda


In [18]:
def tta_predict(model, img_tensor, device, threshold=0.5):
    """TTA с 5 аугментациями + усреднение вероятностей"""
    preds = []
    transforms = [
        lambda x: x,
        lambda x: torch.flip(x, [3]),
        lambda x: torch.rot90(x, k=1, dims=[2,3]),
        lambda x: torch.rot90(x, k=2, dims=[2,3]),
        lambda x: torch.rot90(x, k=3, dims=[2,3])
    ]
    inverses = [
        lambda x: x,
        lambda x: torch.flip(x, [3]),
        lambda x: torch.rot90(x, k=-1, dims=[2,3]),
        lambda x: torch.rot90(x, k=-2, dims=[2,3]),
        lambda x: torch.rot90(x, k=-3, dims=[2,3])
    ]
    with torch.no_grad():
        for t, inv in zip(transforms, inverses):
            aug = t(img_tensor.to(device))
            prob = torch.sigmoid(model(aug)).cpu()
            preds.append(inv(prob))
    prob_map = torch.stack(preds).mean(dim=0)
    mask = (prob_map > threshold).float()
    return prob_map, mask

In [1]:
print(f"CUDA доступно: {torch.cuda.is_available()}")
print(f"Используемое устройство: {device}")
print(f"Выделенная память GPU: {torch.cuda.memory_allocated()/1024**2:.0f} MB")

NameError: name 'torch' is not defined

In [ ]:
random.seed(42)


WEIGHTS_DIR = Path("../src/models")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
best_model_path = WEIGHTS_DIR / "baseline_best.pth"


best_dice = 0.0
patience_counter = 0
max_epochs = 40

for epoch in range(1, max_epochs + 1):
    model.train()
    train_loss = 0
    with tqdm(train_loader, desc=f"Epoch {epoch:02d}/{max_epochs} [Train]", leave=False) as pbar:
        for batch in pbar:
            x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        with tqdm(val_loader, desc=f"Epoch {epoch:02d}/{max_epochs} [Val]  ", leave=False) as pbar:
            for batch in pbar:
                x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
                logits = model(x)
                val_loss += criterion(logits, y).item()

                pred = torch.sigmoid(logits) > 0.5
                pred = pred.to(device).float()
                tp += (pred * y).sum().item()
                fp += (pred * (1 - y)).sum().item()
                fn += ((1 - pred) * y).sum().item()
                tn += ((1 - pred) * (1 - y)).sum().item()


    val_loss /= len(val_loader)
    dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
    iou  = tp / (tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    scheduler.step(dice)

    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | LR: {optimizer.param_groups[0]['lr']:.1e}")

    if dice > best_dice:
      best_dice = dice
      best_val_loss = val_loss
      torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': dice,
            'val_loss': val_loss,
            'IoU': iou,
            'precision': precision,
            'recall': recall,
        }, best_model_path)
      print(f"💾 Сохранена лучшая модель | Dice: {dice:.4f} | Path: {best_model_path}")
      patience_counter = 0
    else:
      patience_counter += 1
      if patience_counter >= 8:
            print("Early stopping triggered")
            break

print(f" Обучение завершено. Лучший Val Dice: {best_dice:.4f}")

In [25]:
rows = []
for split in ["train", "validation"]:
    img_dir = DATASET_ROOT / split / "images"
    mask_dir = DATASET_ROOT / split / "labels"
    for img_p in sorted(img_dir.glob("*.png")):
        mask_p = mask_dir / img_p.name
        if mask_p.exists():
            mask = cv2.imread(str(mask_p), cv2.IMREAD_GRAYSCALE)
            wound_ratio = (mask > 127).mean()
            rows.append({"img": str(img_p), "mask": str(mask_p), "ratio": wound_ratio})

df = pd.DataFrame(rows)

def size_bin(r):
    if r < 0.003: return 0  # маленькие (<0.3%)
    elif r <= 0.016: return 1  # средние (0.3–1.6%)
    else: return 2  # большие (>1.6%)
df["bin"] = df["ratio"].apply(size_bin)
df

,img,mask,ratio,bin
0,..\data\raw\Foot Ulcer Segmentation Challenge\...,..\data\raw\Foot Ulcer Segmentation Challenge\...,0.035385,2
1,..\data\raw\Foot Ulcer Segmentation Challenge\...,..\data\raw\Foot Ulcer Segmentation Challenge\...,0.005238,1
2,..\data\raw\Foot Ulcer Segmentation Challenge\...,..\data\raw\Foot Ulcer Segmentation Challenge\...,0.016571,2
3,..\data\raw\Foot Ulcer Segmentation Challenge\...,..\data\raw\Foot Ulcer Segmentation Challenge\...,0.021812,2
4,..\data\raw\Foot Ulcer Segmentation Challenge\...,..\data\raw\Foot Ulcer Segmentation Challenge\...,0.001591,0
...,...,...,...,...
1005,..\data\raw\Foot Ulcer Segmentation Challenge\...,..\data\raw\Foot Ulcer Segmentation Challenge\...,0.005241,1
1006,..\data\raw\Foot Ulcer Segmentation Challenge\...,..\data\raw\Foot Ulcer Segmentation Challenge\...,0.002174,0
1007,..\data\raw\Foot Ulcer Segmentation Challenge\...,..\data\raw\Foot Ulcer Segmentation Challenge\...,0.031338,2
1008,..\data\raw\Foot Ulcer Segmentation Challenge\...,..\data\raw\Foot Ulcer Segmentation Challenge\...,0.003712,1


In [39]:
random.seed(42)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
out_dir = Path("../data/processed/")
out_dir.mkdir(exist_ok=True)

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df["bin"])):
    fold_dir = Path(out_dir / f"fold_{fold+1}")
    fold_dir.mkdir(exist_ok=True)
    df.iloc[train_idx].to_csv(fold_dir / "train.csv", index=False)
    df.iloc[val_idx].to_csv(fold_dir / "val.csv", index=False)
    print(f"✅ Fold {fold+1}: train={len(train_idx)}, val={len(val_idx)}")

✅ Fold 1: train=808, val=202
✅ Fold 2: train=808, val=202
✅ Fold 3: train=808, val=202
✅ Fold 4: train=808, val=202
✅ Fold 5: train=808, val=202


In [49]:
class FUSegFoldDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img = np.array(Image.open(row["img"]).convert("RGB"))
        mask = np.array(Image.open(row["mask"]).convert('L')) > 127

        if self.transform:
            aug = self.transform(image=img, mask=mask.astype(np.uint8))
            img, mask = aug["image"], aug["mask"]

        img = torch.from_numpy(img.transpose(2,0,1)).float()
        mask = torch.from_numpy(mask).long()
        return {"image": img, "mask": mask}

In [ ]:
random.seed(42)


fold_results = []


for fold in range(1, 6):
    print(f"\n === FOLD {fold}/5 ===")
    
    train_csv = out_dir / f"fold_{fold}" / "train.csv"
    val_csv   = out_dir / f"fold_{fold}" / "val.csv"

    train_ds_fold = FUSegFoldDataset(train_csv, transform=train_tf)
    val_ds_fold   = FUSegFoldDataset(val_csv, transform=val_tf)
    
    train_loader_fold = DataLoader(train_ds_fold, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
    val_loader_fold   = DataLoader(val_ds_fold, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

    model = smp.Unet(encoder_name='timm-efficientnet-b0', encoder_weights='imagenet', in_channels=3, classes=1, activation=None).to(device)
    criterion = DiceFocalLoss(
        dice_weight=1.0,      # Dice — основная метрика
        focal_weight=0.5,     # Focal — вспомогательная стабилизация
        focal_gamma=2.0,      # Стандартное значение из оригинальной статьи
        focal_alpha=0.25      # Усиливает штраф за пропуск раны (класс 1)
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=4, factor=0.5)

    best_dice = 0.0
    patience_counter = 0
    best_path = WEIGHTS_DIR / f"fold_{fold}_best.pth"

    for epoch in range(1, max_epochs + 1):
        
        model.train()
        train_loss = 0.0
        with tqdm(train_loader_fold, desc=f"Fold {fold} | Epoch {epoch} [Train]", leave=False) as pbar:
            for batch in pbar:
                x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
                optimizer.zero_grad()
                loss = criterion(model(x), y)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                train_loss += loss.item()
                pbar.set_postfix(loss=f"{loss.item():.4f}")
        train_loss /= len(train_loader)

        model.eval()
        val_loss, tp, fp, fn, tn = 0.0, 0, 0, 0, 0
        with torch.no_grad():
            with tqdm(val_loader_fold, desc=f"Fold {fold} | Epoch {epoch} [Val]  ", leave=False) as pbar:
                for batch in pbar:
                    x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
                    logits = model(x)
                    val_loss += criterion(logits, y).item()

                    pred = torch.sigmoid(logits) > 0.5
                    pred = pred.to(device).float()
                    tp += (pred * y).sum().item()
                    fp += (pred * (1 - y)).sum().item()
                    fn += ((1 - pred) * y).sum().item()
                    tn += ((1 - pred) * (1 - y)).sum().item()

        val_loss /= len(val_loader)
        dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
        iou  = tp / (tp + fp + fn + 1e-6)
        precision = tp / (tp + fp + 1e-6)
        recall = tp / (tp + fn + 1e-6)
        scheduler.step(dice)

        print(f"Epoch {epoch:02d} | Train L: {train_loss:.4f} | Val L: {val_loss:.4f} | "
              f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | LR: {optimizer.param_groups[0]['lr']:.1e}")

        # Early Stopping
        if dice > best_dice:
            best_dice = dice
            torch.save({'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'val_dice': dice,
                        'IoU': iou,
                        'val_precision': precision,
                        'val_recall': recall}, 
                        best_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 8:
                print("Early stopping triggered")
                break

    fold_results.append({'fold': fold, 'best_dice': best_dice, 'path': str(best_path)})
    print(f"Fold {fold} завершён. Лучший Dice: {best_dice:.4f}")


print("\n5-FOLD CV SUMMARY")
dices = [r['best_dice'] for r in fold_results]
for r in fold_results: print(f"Fold {r['fold']}: {r['best_dice']:.4f}")
print(f"Mean Dice: {np.mean(dices):.4f} ± {np.std(dices):.4f}")

In [65]:
model = smp.Unet(
    encoder_name="mit_b1",          # Mix Transformer (SegFormer backbone)
    encoder_weights="imagenet",
    in_channels=3, classes=1, activation=None
).to(device)

print(f"✅ MiT U-Net загружен. Параметры: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

✅ MiT U-Net загружен. Параметры: 16.4M


In [66]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

criterion = DiceFocalLoss(dice_weight=1.0, 
                          focal_weight=0.5,
                          focal_gamma=2.0, 
                          focal_alpha=0.25)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=5, factor=0.5)

In [69]:
random.seed(42)

best_dice = 0.0
patience_counter = 0

best_model_path = WEIGHTS_DIR / "baseline_mit_best.pth"

for epoch in range(1, max_epochs + 1):
    model.train()
    train_loss = 0
    with tqdm(train_loader, desc=f"Epoch {epoch:02d}/{max_epochs} [Train]", leave=False) as pbar:
        for batch in pbar:
            x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        with tqdm(val_loader, desc=f"Epoch {epoch:02d}/{max_epochs} [Val]  ", leave=False) as pbar:
            for batch in pbar:
                x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
                logits = model(x)
                val_loss += criterion(logits, y).item()

                pred = torch.sigmoid(logits) > 0.5
                pred = pred.to(device).float()
                tp += (pred * y).sum().item()
                fp += (pred * (1 - y)).sum().item()
                fn += ((1 - pred) * y).sum().item()
                tn += ((1 - pred) * (1 - y)).sum().item()

    val_loss /= len(val_loader)
    dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
    iou  = tp / (tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)

    scheduler.step(dice)


    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | LR: {optimizer.param_groups[0]['lr']:.1e}")
    if dice > best_dice:
        best_dice = dice
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': dice,
            'val_loss': val_loss,
            'IoU': iou,
            'precision': precision,
            'recall': recall,
        }, best_model_path)
        print(f"💾 Сохранена лучшая модель | Dice: {dice:.4f} | Path: {best_model_path}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 8:
            print("Early stopping triggered")
            break

print(f"\n Обучение завершено. Лучший Val Dice: {best_dice:.4f}")

Epoch 01 | Train Loss: 0.9902 | Val Loss: 0.9835 | Dice: 0.7082 | IoU: 0.5482 | Precision: 0.6734 | Recall: 0.7466 | LR: 1.0e-04
💾 Сохранена лучшая модель | Dice: 0.7082 | Path: ..\src\models\baseline_mit_best.pth


Epoch 02 | Train Loss: 0.9813 | Val Loss: 0.9766 | Dice: 0.7686 | IoU: 0.6242 | Precision: 0.8098 | Recall: 0.7315 | LR: 1.0e-04
💾 Сохранена лучшая модель | Dice: 0.7686 | Path: ..\src\models\baseline_mit_best.pth


Epoch 03 | Train Loss: 0.9790 | Val Loss: 0.9763 | Dice: 0.7719 | IoU: 0.6286 | Precision: 0.7000 | Recall: 0.8604 | LR: 1.0e-04
💾 Сохранена лучшая модель | Dice: 0.7719 | Path: ..\src\models\baseline_mit_best.pth


Epoch 04 | Train Loss: 0.9767 | Val Loss: 0.9741 | Dice: 0.7132 | IoU: 0.5542 | Precision: 0.8603 | Recall: 0.6090 | LR: 1.0e-04


Epoch 05 | Train Loss: 0.9762 | Val Loss: 0.9726 | Dice: 0.8163 | IoU: 0.6897 | Precision: 0.7454 | Recall: 0.9021 | LR: 1.0e-04
💾 Сохранена лучшая модель | Dice: 0.8163 | Path: ..\src\models\baseline_mit_best.pth


...

Epoch 36 | Train Loss: 0.9725 | Val Loss: 0.9679 | Dice: 0.8992 | IoU: 0.8169 | Precision: 0.8545 | Recall: 0.9488 | LR: 5.0e-05


Epoch 37 | Train Loss: 0.9722 | Val Loss: 0.9677 | Dice: 0.9057 | IoU: 0.8276 | Precision: 0.8933 | Recall: 0.9183 | LR: 5.0e-05


Epoch 38 | Train Loss: 0.9722 | Val Loss: 0.9677 | Dice: 0.9097 | IoU: 0.8344 | Precision: 0.9027 | Recall: 0.9168 | LR: 5.0e-05


Epoch 39 | Train Loss: 0.9719 | Val Loss: 0.9677 | Dice: 0.9085 | IoU: 0.8323 | Precision: 0.9147 | Recall: 0.9024 | LR: 5.0e-05


Epoch 40 | Train Loss: 0.9718 | Val Loss: 0.9677 | Dice: 0.9066 | IoU: 0.8291 | Precision: 0.8784 | Recall: 0.9366 | LR: 5.0e-05

 Обучение завершено. Лучший Val Dice: 0.9145


In [73]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = smp.Unet(encoder_name='timm-efficientnet-b0', encoder_weights='imagenet', in_channels=3, classes=1, activation=None, attention_type='scse').to(device)
criterion = DiceFocalLoss(
    dice_weight=1.0, 
    focal_weight=0.5,     
    focal_gamma=2.0,      
    focal_alpha=0.25      
)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=4, factor=0.5)

In [74]:
random.seed(42)


best_dice = 0.0
patience_counter = 0
best_model_path = WEIGHTS_DIR / "baseline_scse_best.pth"


for epoch in range(1, max_epochs + 1):
    model.train()
    train_loss = 0
    with tqdm(train_loader, desc=f"Epoch {epoch:02d}/{max_epochs} [Train]", leave=False) as pbar:
        for batch in pbar:
            x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        with tqdm(val_loader, desc=f"Epoch {epoch:02d}/{max_epochs} [Val]  ", leave=False) as pbar:
            for batch in pbar:
                x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
                logits = model(x)
                val_loss += criterion(logits, y).item()

                pred = torch.sigmoid(logits) > 0.5
                pred = pred.to(device).float()
                tp += (pred * y).sum().item()
                fp += (pred * (1 - y)).sum().item()
                fn += ((1 - pred) * y).sum().item()
                tn += ((1 - pred) * (1 - y)).sum().item()

    val_loss /= len(val_loader)
    dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
    iou  = tp / (tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)

    scheduler.step(dice)


    print(f"✅ Epoch {epoch:02d} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | LR: {optimizer.param_groups[0]['lr']:.1e}")
    if dice > best_dice:
      best_dice = dice
      best_val_loss = val_loss
      torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': dice,
            'val_loss': val_loss,
            'IoU': iou,
            'precision': precision,
            'recall': recall,
        }, best_model_path)
      print(f"💾 Сохранена лучшая модель | Dice: {dice:.4f} | Path: {best_model_path}")
      patience_counter = 0
    else:
      patience_counter += 1

    if patience_counter >= 8:
            print("⏹️ Early stopping triggered")
            break

print(f"\n🏆 Обучение завершено. Лучший Val Dice: {best_dice:.4f}")

✅ Epoch 01 | Train Loss: 0.9863 | Val Loss: 0.9791 | Dice: 0.5206 | IoU: 0.3519 | Precision: 0.3573 | Recall: 0.9591 | LR: 2.0e-04
💾 Сохранена лучшая модель | Dice: 0.5206 | Path: ..\src\models\baseline_scse_best.pth


✅ Epoch 02 | Train Loss: 0.9752 | Val Loss: 0.9711 | Dice: 0.7836 | IoU: 0.6442 | Precision: 0.6978 | Recall: 0.8935 | LR: 2.0e-04
💾 Сохранена лучшая модель | Dice: 0.7836 | Path: ..\src\models\baseline_scse_best.pth


✅ Epoch 03 | Train Loss: 0.9738 | Val Loss: 0.9708 | Dice: 0.7997 | IoU: 0.6663 | Precision: 0.8114 | Recall: 0.7884 | LR: 2.0e-04
💾 Сохранена лучшая модель | Dice: 0.7997 | Path: ..\src\models\baseline_scse_best.pth


✅ Epoch 04 | Train Loss: 0.9732 | Val Loss: 0.9704 | Dice: 0.8039 | IoU: 0.6722 | Precision: 0.9014 | Recall: 0.7255 | LR: 2.0e-04
💾 Сохранена лучшая модель | Dice: 0.8039 | Path: ..\src\models\baseline_scse_best.pth


✅ Epoch 05 | Train Loss: 0.9726 | Val Loss: 0.9695 | Dice: 0.8449 | IoU: 0.7314 | Precision: 0.8810 | Recall: 0.8116 | LR: 2.0e-04
💾 Сохранена лучшая модель | Dice: 0.8449 | Path: ..\src\models\baseline_scse_best.pth


...

✅ Epoch 20 | Train Loss: 0.9710 | Val Loss: 0.9679 | Dice: 0.8797 | IoU: 0.7852 | Precision: 0.8190 | Recall: 0.9500 | LR: 1.0e-04


✅ Epoch 21 | Train Loss: 0.9711 | Val Loss: 0.9678 | Dice: 0.8952 | IoU: 0.8103 | Precision: 0.8951 | Recall: 0.8953 | LR: 5.0e-05


✅ Epoch 22 | Train Loss: 0.9708 | Val Loss: 0.9677 | Dice: 0.8964 | IoU: 0.8123 | Precision: 0.8729 | Recall: 0.9213 | LR: 5.0e-05


✅ Epoch 23 | Train Loss: 0.9704 | Val Loss: 0.9678 | Dice: 0.8892 | IoU: 0.8006 | Precision: 0.8447 | Recall: 0.9387 | LR: 5.0e-05


✅ Epoch 24 | Train Loss: 0.9711 | Val Loss: 0.9676 | Dice: 0.8961 | IoU: 0.8117 | Precision: 0.8644 | Recall: 0.9302 | LR: 5.0e-05
⏹️ Early stopping triggered

🏆 Обучение завершено. Лучший Val Dice: 0.8970


In [75]:
model = smp.Unet(
    encoder_name="mit_b1",
    encoder_weights="imagenet",
    in_channels=3, classes=1, activation=None
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

criterion = DiceFocalLoss(dice_weight=1.0, 
                          focal_weight=0.5,
                          focal_gamma=2.0, 
                          focal_alpha=0.25)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=4, factor=0.5)

In [76]:
random.seed(42)

best_dice = 0.0
patience_counter = 0

best_model_path = WEIGHTS_DIR / "baseline_mit_2_best.pth"

for epoch in range(1, max_epochs + 1):
    model.train()
    train_loss = 0
    with tqdm(train_loader, desc=f"Epoch {epoch:02d}/{max_epochs} [Train]", leave=False) as pbar:
        for batch in pbar:
            x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        with tqdm(val_loader, desc=f"Epoch {epoch:02d}/{max_epochs} [Val]  ", leave=False) as pbar:
            for batch in pbar:
                x, y = batch['image'].to(device), batch['mask'].float().unsqueeze(1).to(device)
                logits = model(x)
                val_loss += criterion(logits, y).item()

                pred = torch.sigmoid(logits) > 0.5
                pred = pred.to(device).float()
                tp += (pred * y).sum().item()
                fp += (pred * (1 - y)).sum().item()
                fn += ((1 - pred) * y).sum().item()
                tn += ((1 - pred) * (1 - y)).sum().item()

    val_loss /= len(val_loader)
    dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
    iou  = tp / (tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)

    scheduler.step(dice)


    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | LR: {optimizer.param_groups[0]['lr']:.1e}")
    if dice > best_dice:
        best_dice = dice
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': dice,
            'val_loss': val_loss,
            'IoU': iou,
            'precision': precision,
            'recall': recall,
        }, best_model_path)
        print(f"💾 Сохранена лучшая модель | Dice: {dice:.4f} | Path: {best_model_path}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 8:
            print("Early stopping triggered")
            break

print(f"\n Обучение завершено. Лучший Val Dice: {best_dice:.4f}")

Epoch 01 | Train Loss: 0.9859 | Val Loss: 0.9763 | Dice: 0.4364 | IoU: 0.2791 | Precision: 0.9272 | Recall: 0.2854 | LR: 2.0e-04
💾 Сохранена лучшая модель | Dice: 0.4364 | Path: ..\src\models\baseline_mit_2_best.pth


Epoch 02 | Train Loss: 0.9762 | Val Loss: 0.9721 | Dice: 0.7038 | IoU: 0.5430 | Precision: 0.5643 | Recall: 0.9350 | LR: 2.0e-04
💾 Сохранена лучшая модель | Dice: 0.7038 | Path: ..\src\models\baseline_mit_2_best.pth


Epoch 03 | Train Loss: 0.9751 | Val Loss: 0.9716 | Dice: 0.7182 | IoU: 0.5603 | Precision: 0.5817 | Recall: 0.9384 | LR: 2.0e-04
💾 Сохранена лучшая модель | Dice: 0.7182 | Path: ..\src\models\baseline_mit_2_best.pth


Epoch 04 | Train Loss: 0.9747 | Val Loss: 0.9694 | Dice: 0.8502 | IoU: 0.7395 | Precision: 0.8147 | Recall: 0.8890 | LR: 2.0e-04
💾 Сохранена лучшая модель | Dice: 0.8502 | Path: ..\src\models\baseline_mit_2_best.pth


Epoch 05 | Train Loss: 0.9738 | Val Loss: 0.9816 | Dice: 0.4789 | IoU: 0.3149 | Precision: 0.3164 | Recall: 0.9850 | LR: 2.0e-04


...

Epoch 33 | Train Loss: 0.9723 | Val Loss: 0.9675 | Dice: 0.9007 | IoU: 0.8193 | Precision: 0.8827 | Recall: 0.9194 | LR: 2.0e-04


Epoch 34 | Train Loss: 0.9722 | Val Loss: 0.9685 | Dice: 0.8466 | IoU: 0.7340 | Precision: 0.7982 | Recall: 0.9013 | LR: 1.0e-04


Epoch 35 | Train Loss: 0.9724 | Val Loss: 0.9674 | Dice: 0.9072 | IoU: 0.8302 | Precision: 0.9087 | Recall: 0.9058 | LR: 1.0e-04


Epoch 36 | Train Loss: 0.9727 | Val Loss: 0.9675 | Dice: 0.8961 | IoU: 0.8118 | Precision: 0.8579 | Recall: 0.9379 | LR: 1.0e-04


Epoch 37 | Train Loss: 0.9714 | Val Loss: 0.9675 | Dice: 0.8867 | IoU: 0.7964 | Precision: 0.8253 | Recall: 0.9579 | LR: 1.0e-04
Early stopping triggered

 Обучение завершено. Лучший Val Dice: 0.9073


In [77]:
base_model = torch.load("../src/models/baseline_best.pth")

C:\Users\Андрей\AppData\Local\Temp\ipykernel_6864\2967256969.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  base_model = torch.load("../src/models/baseline_best.pth")


In [89]:
print(f"Эпоха с лучшими значениями: {base_model['epoch']},"
f" Функция потерь: {base_model['val_loss']},"
f" Dice: {base_model['val_dice']},"
f" IoU: {base_model['IoU']},"
f" Precision: {base_model['precision']},"
f" Recall: {base_model['recall']}")

Эпоха с лучшими значениями: 31, Функция потерь: 0.9673600602149963, Dice: 0.9105439862659859, IoU: 0.8357785672733835, Precision: 0.89548000862856, Recall: 0.9261234552475874
